In [58]:


import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
import sklearn

In [59]:
from archive.lstm_anomaly_detector import make_windows, LSTMAnomalyDetector

In [60]:
TIME_STEP = 10
SAMPLE_AGGREGATION_RATE = 20

In [61]:
## Load and scale raw data
door_events = pd.read_pickle("../door_events.pkl")

# Normalise data with non-anomalous points only
normal_mask = door_events["label"] == 0

features = door_events.iloc[:,9:25]
normal_X = features[normal_mask]
scaler = sklearn.preprocessing.StandardScaler()
scaler.fit(normal_X) # Only fit to normal points
trf_X = pd.DataFrame(scaler.transform(features)) # Transform all points

In [62]:
### Resample training data
sampled_X = trf_X.groupby(trf_X.index // SAMPLE_AGGREGATION_RATE).mean()
window_X = make_windows(sampled_X, TIME_STEP)
print(f"window_X shape: {window_X.shape}")
assert window_X.shape[1] == TIME_STEP

### Resample labels
resampled_labels = door_events["label"].groupby(door_events.index // SAMPLE_AGGREGATION_RATE).max()
window_y = make_windows(resampled_labels, TIME_STEP)[:,-1]
print(f"window_y shape: {window_y.shape}")

### Separate normal and anomalous windows
normal_window_mask = window_y == 0
normal_window_X = window_X[normal_window_mask]
anomalous_window_X = window_X[~normal_window_mask]


window_X shape: (1337, 10, 16)
window_y shape: (1337,)


In [63]:
### Build and fit the door-sensor anomaly detector
door_detector = LSTMAnomalyDetector(
    timesteps=TIME_STEP, n_features=window_X.shape[2],
    latent_dim=16, dropout=0.2, name="door_lstm",
    feature_names=list(features.columns),
)

train_X, test_normal_X = train_test_split(normal_window_X, test_size=0.2, shuffle=True, random_state=42)
fit_X, val_X = train_test_split(train_X, test_size=0.2, shuffle=True, random_state=42)

door_detector.fit(fit_X, val_X, epochs=200)

TypeError: LSTMAnomalyDetector.__init__() got an unexpected keyword argument 'feature_names'

In [ ]:
### Split test data into tuning and final testing sets
tuning_normal_X, eval_normal_X = train_test_split(test_normal_X, test_size=0.5, random_state=42)
tuning_anomalous_X, eval_anomalous_X = train_test_split(anomalous_window_X, test_size=0.5, random_state=42)

tuning_X = np.concatenate([tuning_normal_X, tuning_anomalous_X])
tuning_y = np.concatenate([
    np.zeros(len(tuning_normal_X)),
    np.ones(len(tuning_anomalous_X)),
])

eval_X = np.concatenate([eval_normal_X, eval_anomalous_X])
eval_y = np.concatenate([
    np.zeros(len(eval_normal_X)),
    np.ones(len(eval_anomalous_X)),
])

In [ ]:
# Compute MSEs (just for the histogram -- tune_threshold below recomputes this internally)
tuning_mses = door_detector.score(tuning_X)

plt.hist(tuning_mses, bins=40)
print(len(tuning_mses))

In [ ]:
### Tune cutoff across precision, recall, specificity, F1, and F2, then pick by F2
# Missed breakdowns are rare but fatal, so we optimise for F2 (recall-biased) rather than F1.
# Plain recall isn't used directly -- it's trivially maximised by flagging everything as
# anomalous, which would be useless in practice.
best = door_detector.tune_threshold(tuning_X, tuning_y, metric="f2")
print(best)

plt.figure()
for name in ["precision", "recall", "specificity", "f1", "f2"]:
    plt.plot(door_detector.tuning_report_["cutoff"], door_detector.tuning_report_[name], label=name)
plt.xlabel("Percentile cutoff (on tuning_mses)")
plt.ylabel("Score")
plt.legend()
plt.show()

In [ ]:
### Final evaluation on eval_X -- untouched by fitting or threshold tuning
eval_metrics = door_detector.evaluate(eval_X, eval_y)
print(f"Evaluation precision   = {eval_metrics['precision']}")
print(f"Evaluation recall      = {eval_metrics['recall']}")
print(f"Evaluation specificity = {eval_metrics['specificity']}")
print(f"Evaluation F1 score    = {eval_metrics['f1']}")

In [ ]:
### (4) Sweep SAMPLE_AGGREGATION_RATE as a hyperparameter.
### Selected via tuning-set F2 only -- eval_X is never touched here, to avoid
### the same leakage we fixed earlier (a hyperparameter picked by peeking at eval
### makes the final eval score biased).
def run_pipeline_for_rate(aggregation_rate, time_step=TIME_STEP, epochs=100):
    sampled = trf_X.groupby(trf_X.index // aggregation_rate).mean()
    win_X = make_windows(sampled, time_step)

    resampled_lbl = door_events["label"].groupby(door_events.index // aggregation_rate).max()
    win_y = make_windows(resampled_lbl, time_step)[:, -1]

    normal_mask = win_y == 0
    normal_X = win_X[normal_mask]
    anomalous_X = win_X[~normal_mask]

    train_X_, test_normal_X_ = train_test_split(normal_X, test_size=0.2, shuffle=True, random_state=42)
    fit_X_, val_X_ = train_test_split(train_X_, test_size=0.2, shuffle=True, random_state=42)

    detector = LSTMAnomalyDetector(time_step, win_X.shape[2], latent_dim=16, dropout=0.2,
                                    name=f"door_lstm_rate_{aggregation_rate}")
    detector.fit(fit_X_, val_X_, epochs=epochs, verbose=0)

    tuning_normal_X_, _ = train_test_split(test_normal_X_, test_size=0.5, random_state=42)
    tuning_anomalous_X_, _ = train_test_split(anomalous_X, test_size=0.5, random_state=42)

    tuning_X_ = np.concatenate([tuning_normal_X_, tuning_anomalous_X_])
    tuning_y_ = np.concatenate([np.zeros(len(tuning_normal_X_)), np.ones(len(tuning_anomalous_X_))])

    best = detector.tune_threshold(tuning_X_, tuning_y_, metric="f2", verbose=0)
    return {
        "aggregation_rate": aggregation_rate,
        "n_windows": len(win_X),
        "n_anomalous": len(anomalous_X),
        **best,
    }

In [ ]:
### Compare candidate aggregation rates on tuning-set F2
candidate_rates = [10, 15, 20, 25, 30]
rate_results = [run_pipeline_for_rate(rate) for rate in candidate_rates]

rate_comparison = pd.DataFrame(rate_results)
print(rate_comparison)

# Pick whichever row has the best f2, then set SAMPLE_AGGREGATION_RATE to that value
# at the top of the notebook and re-run the full pipeline (including the untouched
# final eval_X evaluation) with it -- don't just trust this sweep's own numbers as final,
# they were only ever measured against tuning_X.

In [ ]:
### (5) Per-feature reconstruction error breakdown on eval_X.
### Aggregate MSE hides *which* of the 16 features actually separate anomalies
### from normal behaviour -- this looks at each feature individually.
feature_breakdown = door_detector.feature_breakdown(eval_X, eval_y)
print(feature_breakdown)

plt.figure()
plt.bar(feature_breakdown["feature_index"].astype(str), feature_breakdown["gap"])
plt.xlabel("Feature index (column in raw_X)")
plt.ylabel("Anomalous MSE - Normal MSE")
plt.title("Which features separate anomalies from normal reconstruction error")
plt.show()

# A large positive gap means that feature's reconstruction error is a strong anomaly
# signal; a gap near zero means the model reconstructs that feature about equally well
# whether the window is normal or anomalous -- i.e. it isn't contributing to detection.
# Features with consistently near-zero gaps are candidates to drop, which would shrink
# the input size and could make the aggregate MSE signal sharper.

In [ ]:
### Example: use the feature diagnostic above to edit the model, then re-fit/tune/evaluate.
# Drop features whose gap is <= 0 -- their reconstruction error doesn't separate
# anomalous from normal windows, so they're just adding noise/capacity for nothing.
print(door_detector.get_architecture())
print(door_detector.list_features())

weak_features = feature_breakdown.loc[feature_breakdown["gap"] <= 0, "feature_name"].tolist()
print("Excluding weak features:", weak_features)

# fit_X/val_X/tuning_X/eval_X are unchanged (still all 16 columns) -- the class
# applies the feature mask internally, so you never need to re-slice arrays by hand.
door_detector.exclude_features(weak_features)   # rebuilds the model with fewer inputs
door_detector.fit(fit_X, val_X, epochs=200)
door_detector.tune_threshold(tuning_X, tuning_y, metric="f2")
print(door_detector.evaluate(eval_X, eval_y))

# Architecture is editable the same way, e.g. trying a smaller bottleneck:
door_detector.set_architecture(latent_dim=8)
door_detector.fit(fit_X, val_X, epochs=200)
door_detector.tune_threshold(tuning_X, tuning_y, metric="f2")
print(door_detector.evaluate(eval_X, eval_y))